In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [2]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [ ]:

esquemaProduction = "Production"

dimensionSubProductCategory = pd.read_sql_table("ProductSubcategory", motorBaseDatos, esquemaProduction)
dimensionSubProductCategory


c:\Users\dange.DANGERPC\OneDrive\Escritorio\etl-aventure-works\my_env\Lib\site-packages\pandas\io\sql.py:1737: SAWarning: Did not recognize type 'Name' of column 'Name'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


,ProductSubcategoryID,ProductCategoryID,Name,rowguid,ModifiedDate
0,1,1,Mountain Bikes,2d364ade-264a-433c-b092-4fcbf3804e01,2008-04-30
1,2,1,Road Bikes,000310c0-bcc8-42c4-b0c3-45ae611af06b,2008-04-30
2,3,1,Touring Bikes,02c5061d-ecdc-4274-b5f1-e91d76bc3f37,2008-04-30
3,4,2,Handlebars,3ef2c725-7135-4c85-9ae6-ae9a3bdd9283,2008-04-30
4,5,2,Bottom Brackets,a9e54089-8a1e-4cf5-8646-e3801f685934,2008-04-30
5,6,2,Brakes,d43ba4a3-ef0d-426b-90eb-4be4547dd30c,2008-04-30
6,7,2,Chains,e93a7231-f16c-4b0f-8c41-c73fdec62da0,2008-04-30
7,8,2,Cranksets,4f644521-422b-4f19-974a-e3df6102567e,2008-04-30
8,9,2,Derailleurs,1830d70c-aa2a-40c0-a271-5ba86f38f8bf,2008-04-30
9,10,2,Forks,b5f9ba42-b69b-4fdd-b2ec-57fb7b42e3cf,2008-04-30


TRANSFORMACION

In [6]:


dimensionSubProductCategory.rename(columns={
    'ProductSubcategoryID': 'ProductSubcategoryKey',
    'Name' : 'EnglishProductSubcategoryName',
    'ProductCategoryID' : 'ProductCategoryKey'

}, inplace=True)

dimensionSubProductCategory["ProductSubcategoryAlternateKey"] = dimensionSubProductCategory["ProductSubcategoryKey"]
dimensionSubProductCategory["SpanishProductCategoryName"] = None
dimensionSubProductCategory["FrenchProductCategoryName"] = None


dimensionSubProductCategory.drop(columns=[
    'rowguid',
    'ModifiedDate',
], inplace=True)

dimensionSubProductCategory

,ProductSubcategoryKey,ProductCategoryKey,EnglishProductSubcategoryName,ProductSubcategoryAlternateKey,SpanishProductCategoryName,FrenchProductCategoryName
0,1,1,Mountain Bikes,1,None,None
1,2,1,Road Bikes,2,None,None
2,3,1,Touring Bikes,3,None,None
3,4,2,Handlebars,4,None,None
4,5,2,Bottom Brackets,5,None,None
5,6,2,Brakes,6,None,None
6,7,2,Chains,7,None,None
7,8,2,Cranksets,8,None,None
8,9,2,Derailleurs,9,None,None
9,10,2,Forks,10,None,None


CARGAR A LA BODEGA

In [7]:
dimensionSubProductCategory.to_sql('dimensionProductSubCategory',motorBodegaDatos, if_exists='replace',index=False)

37